In [ ]:
# Bank Data Exploration

## 1. Raw data inspection
## 2. Data quality issues found
## 3. Cleaning decisions made
## 4. Category mapping approach
## 5. Final clean dataset verification

## This notebook explores the data from Eurobank transaction statements.
### The goal is to clean and combine data from different years and create a parquet file containing everything, for future analysis.

In [ ]:
import json
import pandas as pd
from pathlib import Path
from IPython.display import display

In [ ]:
PROJECT_PATH = Path.cwd().parent
ARCHIVE_PATH = PROJECT_PATH / "archive"
DATA_PATH = PROJECT_PATH / "data"
MAPPINGS_PATH = PROJECT_PATH / "mappings"

print(PROJECT_PATH)
print(ARCHIVE_PATH)
print(DATA_PATH)
print(MAPPINGS_PATH)

#### --- DATA INSPECTION

In [ ]:
def inspect_datasets(bank: str) -> list[Path]:
    """
    Inspect raw CSV datasets for a given bank and return their file paths.
    
    Displays the first two and last rows, column types, and NaN counts for each CSV file found in the bank's raw data directory.

    Args:
        bank: Bank identifier matching the subdirectory and file prefix
              under data/raw/ (e.g. "eurobank", "revolut").

    Returns:
        Sorted list of Path objects pointing to the matched CSV files.

    Raises:
    """
    
    datasets = sorted((DATA_PATH / "raw" / bank).glob(f"{bank}_*.csv"))

    for d in datasets:
        df = pd.read_csv(d,sep=";")
        print(f"---------- DATASET # {d.name}")
        display(df.head(2))
        display(df.tail())
        print("Types: ")
        display(df.dtypes)
        print("Nans: ")
        display(df.isna().sum().to_dict())
        print("\n\n\n")

    return datasets

EUROBANK_DATASETS = inspect_datasets("eurobank")

#### --- CLEANING DECISIONS

Upon inspection we conclude to the following findings:
1. columns names are in greek --> need to be renamed
2. amounts do not follow the european number formatting --> need to be changed
3. "Ονοματεπώνυμο/Επωνυμία Αντισυμβαλλόμενου" column does not contain any info --> will be dropped
4. "ΗΜ/ΝΙΑ ΑΞΙΑS" columns not useful --> will be dropped
5. amount should be converted to float type
6. date should be converted to datetime type
7. the last 4 rows of all datasets do not contain transaction data --> will be dropped

In [ ]:
def clean_data_eurobank(datasets: list[Path]) -> list[pd.DataFrame]:
    """
    Read and clean Eurobank CSV files into a list of DataFrames.

    For each CSV:
    1. drops unnecessary columns, 
    2. renames Greek headers to English, 
    3. parses dates with day-first format, 
    4. converts European number formatting (dot as thousands separator, comma as decimal) to floats, 
    5. adds a source column for traceability. 
    6. kips the last 4 rows of each file (Eurobank footer metadata).

    Args:
        datasets: List of Path objects pointing to raw Eurobank CSV files.

    Returns:
        List of cleaned DataFrames, one per input file, with columns: date, description, amount, balance, source.

    Raises:
    """

    # List to add the dataframes
    dfs = []
    
    for d in datasets:
        df = pd.read_csv(d, skipfooter=4, engine="python", sep=";")
        df = df.drop(["ΗΜ/ΝΙΑ ΑΞΙΑΣ", "Ονοματεπώνυμο/Επωνυμία Αντισυμβαλλόμενου"], axis=1)
        df = df.rename(columns = {
            "ΗΜ/ΝΙΑ ΚΙΝΗΣΗΣ": "date",
            "ΠΕΡΙΓΡΑΦΗ": "description",
            "ΠΟΣΟ": "amount",
            "ΥΠΟΛΟΙΠΟ": "balance",
        })
        df["date"] = pd.to_datetime(df["date"], dayfirst=True)
        
        for col in ["amount", "balance"]:
            df[col] = (
                df[col]
                .str.replace(".", "", regex=False)
                .str.replace(",", ".", regex=False)
                .astype(float)
            )

        # New column 'source' will help us identify the origin of each data entry.
        # Useful for identification when we create the master db with all the datasets from all the banks.
        df["source"] = d.name

        print(f"--- {d.name}, rows: {len(df)}")
        print(f"NaNs: {df.isna().sum().to_dict()}")

        dfs.append(df)

    return dfs

DFS = clean_data_eurobank(EUROBANK_DATASETS)

#### --- DESCRIPTION TO CATEGORY MAPPING

The next important step is to map each description to a specific category,
e.g "Lidl" --> "Market".
The reason this is needed is becaused "category" column will help us produce aggragate results on spendings and incomes.

Steps: 
1. Merge the dataframes
2. Extract unique descriptions
3. Write and save the descriptions into batches of json files --> save it in "./archive/batch_blank_mappings"
4. Externally, put each json file into an AI and ask it to map whatever it recognizes, and mark as "misc" whatever it does not. --> save it in "./archive/batch_ai_mappings"
5. Merge the AI generated json files into one master json file --> save it in "./mappings/ai"
6. Open the master json file in a text editor and try to categorize as many "misc" as you can --> save it in "./mappings/completed"
7. Once you are happy with the result, map the master df with the description to category mapping into the "category" column.
8. Save it into a new parquet file.

In [ ]:
# Step 1: Merge the dataframes

def merge_dfs(dfs: list[pd.DataFrame]) -> pd.DataFrame:
    df = pd.concat(dfs, ignore_index=True)
    print(f"The new dataframe has {len(df)} rows and these columns: {list(df.columns)}")
    return df

DF = merge_dfs(DFS)

In [ ]:
# Steps 2 and 3: Extract unique descriptions & write and save the descriptions into batches of json files

def save_descriptions_in_batches(df: pd.DataFrame, batch_size: int, bank) -> None:
    """
    Extract unique descriptions from a DataFrame and save them as batched JSON files.

    Each file maps description strings to "blank", producing a template for AI-assisted category mapping.
    Batching limits the number of descriptions per file to reduce errors during the AI categorization process.

    Example output entry: {"Lidl": "blank", "Doctor": "blank", "S.F.CHAI GR": "blank"}

    Args:
        df: DataFrame containing a 'description' column.
        batch_size: Maximum number of descriptions per output file.
        bank: Bank identifier used in output filenames and directory structure.

    Raises:
    """
    
    out_dir = ARCHIVE_PATH / "blank_mappings" / bank
    out_dir.mkdir(exist_ok=True)
    
    descriptions = sorted(df["description"].dropna().unique())

    for i in range(0, len(descriptions), batch_size):

        batch = {k: "blank" for k in descriptions[i : i + batch_size]}
        batch_num = (i // batch_size) + 1
        
        out_file = out_dir / f"{bank}_blank_description_mapping_{batch_num:02d}.json"

        with open(out_file, "w", encoding="utf-8") as f:
            json.dump(batch, f, ensure_ascii=False, indent=4)

        print(f"Created {out_file.name} ({len(batch)} descriptions)")
    
    
save_descriptions_in_batches(DF, 200, "eurobank")

In [ ]:
# Step 5

def merge_batch_ai_mappings(bank) -> None:
    """
    Merge AI-categorized batch JSON files into a single master mapping.

    Reads all batch files from archive/batch_ai_mappings/{bank}/, combines them into one dictionary, and writes the result
    to mappings/ai/{bank}.json. Prints a summary of how many descriptions remain categorized as "misc".

    Example output entry:
        {"Lidl": "Market", "Doctor": "Medical", "S.F.CHAI GR": "misc"}

    Args:
        bank: Bank identifier matching the subdirectory and file prefix (e.g. "eurobank", "revolut").

    Raises:
    """
    
    ai_mappings_files = sorted((ARCHIVE_PATH / "ai_mappings" / bank).glob(f"{bank}_*.json"))
    ai_mappings = dict()

    for file in ai_mappings_files:
        with open(file, "r", encoding="utf-8") as f:
            ai_mappings.update(json.load(f))

    with open(MAPPINGS_PATH / "ai" / f"{bank}.json", "w", encoding="utf-8") as f:
        json.dump(ai_mappings, f, ensure_ascii=False, indent=4)
        
    miscs = sum(1 for v in ai_mappings.values() if v == "misc")

    print(f"There are {miscs} 'misc' labeled descriptions left to categorize, out of {len(ai_mappings)} unique descriptions.")

merge_batch_ai_mappings("eurobank")

In [ ]:
def sanity_check(df: pd.DataFrame, bank: str):

    descr_in_df = set(df["description"].dropna().unique())

    with open(MAPPINGS_PATH / "completed" / f"{bank}.json", encoding="utf-8") as f:
        descr_in_completed = set(json.load(f))

    with open(MAPPINGS_PATH / "ai" / f"{bank}.json", encoding="utf-8") as f:
        descr_in_ai = set(json.load(f))

    print(f"Unique descriptions in the Data Frame: {len(descr_in_df)}")
    print(f"Unique descriptions in the completed mappings json: {len(descr_in_completed)}")
    print(f"Unique descriptions in the ai mappings json: {len(descr_in_ai)}")

sanity_check(DF, "eurobank")

In [ ]:
def map_descriptions_to_categories(df: pd.DataFrame, bank: str) -> None:
    
    with open(MAPPINGS_PATH / "completed" / f"{bank}.json", encoding="utf-8") as f:
        mappings = json.load(f)

    df["category"] = df["description"].map(mappings)

    return df

DF = map_descriptions_to_categories(DF, "eurobank")

In [ ]:
# Sanity check
print(f"Total rows: {len(DF)}", end='\n\n')
print(f'Min date: {DF["date"].min()}', end='\n\n')
print(f'Max date: {DF["date"].max()}', end='\n\n')
print(f'NaNs: {DF.isna().sum().to_dict()}', end='\n\n')
print(f'Categories: {DF["category"].nunique()} unique', end='\n\n')
print(f"\nCategory breakdown:")
print(DF["category"].value_counts().sort_index().to_string())

In [ ]:
def save_to_parquet(df: pd.DataFrame, name: str) -> None:
    out_dir = DATA_PATH / "processed"
    out_dir.mkdir(parents=True, exist_ok=True)
    df.to_parquet(out_dir / name)

save_to_parquet(DF, "eurobank_2021-06_2026-05.parquet")